# Reading Our Staged Datasets With Autoloaders


In [0]:
class Clean_SilverDims:
    def __init__(self, df=None):
        self.df = df

    def drop_rescue_data(self, *cols):
        self.df = self.df.drop("_rescued_data")
        return self.df
   
    def read_data(self, folder_name):
        self.df = spark.readStream.format("cloudFiles") \
            .option("cloudFiles.format","parquet") \
            .option("cloudFiles.schemaLocation", f"abfss://silver@jessdatalake.dfs.core.windows.net/{folder_name}/checkpoint_location") \
            .option("schemaEvolutionMode", "addNewColumns") \
            .load(f"abfss://bronze@jessdatalake.dfs.core.windows.net/{folder_name}")
        return self.df
    
    def write_data(self, folder_name, data_name):
        self.df.writeStream.format("delta") \
            .outputMode("append") \
            .option("checkpointLocation", f"abfss://silver@jessdatalake.dfs.core.windows.net/{folder_name}/checkpoint_location") \
            .option("path", f"abfss://silver@jessdatalake.dfs.core.windows.net/{folder_name}/{data_name}") \
            .trigger(once=True) \
            .toTable(f"musicstreaming_project.silver.{data_name}")
        return self.df
    
    def drop_dup(self, primary_key):
        self.df = self.df.dropDuplicates([primary_key])
        return self.df


## DimArtist Ingestion/Transformations with Autoloaders

In [0]:
Clean_Artist = Clean_SilverDims()
df_artist = Clean_Artist.read_data("DimArtist")
df_artist = Clean_Artist.drop_dup("artist_id")
df_artist = Clean_Artist.drop_rescue_data()
Clean_Artist.write_data("DimArtist", "dimartist")
df_artist.display()

## DimTrack Ingestion/Transformations with Autoloaders

In [0]:
Clean_Track = Clean_SilverDims()
df_track = Clean_Track.read_data("DimTrack")
df_track = Clean_Track.drop_dup("track_id")
df_track = Clean_Track.drop_rescue_data()
df_track.display()


In [0]:
df_track = df_track.withColumn("duration_flag", when(col("duration_sec")< 150, "short")\
    .when(col("duration_sec")>=150, "medium")\
    .otherwise("long"))

df_track.display()

In [0]:
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))
df_track.display()

In [0]:
Clean_Track.write_data("DimTrack", "dimtrack")

## DimUser Ingestion/Transformations with Autoloaders

In [0]:
Clean_User = Clean_SilverDims()
df_user = Clean_User.read_data("DimUser")
df_user = Clean_User.drop_dup("user_id")
df_user = Clean_User.drop_rescue_data()
df_user.display()


In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn("subscription_type", regexp_replace("subscription_type", "Premium", "Premium Plan"))\
    .withColumn("subscription_type", regexp_replace("subscription_type", "Free", "Free Plan"))\
    .withColumn("subscription_type", regexp_replace("subscription_type", "Family", "Family Plan"))


In [0]:
Clean_User.write_data("DimUser", "dimuser")

## DimDate Ingestion/Transformations with Autoloaders

In [0]:
Clean_Date = Clean_SilverDims()
df_date = Clean_Date.read_data("DimDate")
df_date = Clean_Date.drop_dup("date_id")
df_date = Clean_Date.drop_rescue_data()
df_date.display()


In [0]:
Clean_Date.write_data("DimDate", "dimdate")

## FactStreams Ingestion/Transformations with Autoloaders

In [0]:
Clean_Fact = Clean_SilverDims()
df_fact = Clean_Fact.read_data("FactStream")
df_fact = Clean_Fact.drop_dup("stream_id")
df_fact = Clean_Fact.drop_rescue_data()
df_fact.displya()

In [0]:
Clean_Fact.write_data("FactStream", "factstream")

In [0]:
%sql
SELECT * FROM musicstreaming_project.gold.dimuser